# Setup Inicial

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('dataset_evasao_escolar_500_registros_com_valores_faltantes.csv')

In [3]:
def generate_gaussian_distribution(series, x_label, y_label, title):
    sns.histplot(series, bins=20, kde=True)
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    plt.show()

In [4]:
def get_invalid_numbers(series):
    numeric = pd.to_numeric(series, errors="coerce")
    invalids = series[numeric.isna() & series.notna()]
    return invalids

# Compreensão inicial dos dados

a) Apresente um breve relatório contendo as seguintes informações: número de registros, número de atributos, nomes das variáveis, tipos de dados e estatísticas descritivas das variáveis numéricas (média, mediana e moda), verificando se estas variáveis possuem distribuição normal.



Antes das mudanças feitas no dataset ao longo do arquivo:

| Variável | Tipo de dado | Registros não nulos | Média | Mediana | Moda |
|---|---|---:|---:|---:|---|
| `id_aluno` | str | 500 | — | — | — |
| `idade` | str | 488 | — | — | 17 |
| `escola_origem` | str | 500 | — | — | Pública |
| `municipio_residencia` | str | 488 | — | — | Maceió |
| `estado_residencia` | str | 500 | — | — | AL |
| `renda_familiar_mensal` | str | 478 | — | — | 700,00 |
| `pessoas_domicilio` | str | 500 | — | — | 5 |
| `acesso_internet` | str | 486 | — | — | Sim |
| `trabalha` | str | 500 | — | — | Não |
| `turno` | str | 500 | — | — | Noite |
| `horas_estudo_semana` | str | 483 | — | — | 10,9 |
| `frequencia_percentual` | str | 482 | — | — | 100,0 |
| `nota_media` | str | 484 | — | — | 6,9 |
| `reprovacoes` | str | 500 | — | — | 0 |
| `data_matricula` | str | 488 | — | — | 2026-02-20 |
| `situacao` | str | 500 | — | — | Ativo |
| `motivo_evasao` | str | 102 | — | — | Problemas de transporte |


Perceba que não dá para calcular a média e mediana, pois todas as colunas são objects, o que impede de fazer cálculos com elas.

Após a limpeza ser feita no dataset:

| Variável | Tipo de dado | Registros não nulos | Média | Mediana | Moda |
|---|---|---:|---:|---:|---|
| `id_aluno` | str | 456 | — | — | — |
| `idade` | int64 | 456 | 20,59 | 20 | 17 |
| `escola_origem` | string | 456 | — | — | Pública |
| `municipio_residencia` | string | 456 | — | — | Maceió |
| `estado_residencia` | str | 456 | — | — | AL |
| `renda_familiar_mensal` | float64 | 456 | 3.162,59 | 2.766,26 | 3.162,59 |
| `pessoas_domicilio` | int64 | 456 | 4,10 | 4 | 4 |
| `acesso_internet` | string | 456 | — | — | Sim |
| `trabalha` | str | 456 | — | — | Não |
| `turno` | string | 456 | — | — | Noite |
| `horas_estudo_semana` | float64 | 456 | 8,41 | 8,40 | 8,4 |
| `frequencia_percentual` | float64 | 456 | 75,07 | 80,25 | 20,0 |
| `nota_media` | float64 | 456 | 7,12 | 7,20 | 7,2 |
| `reprovacoes` | int64 | 456 | 0,98 | 1 | 0 |
| `data_matricula` | datetime64[us] | 442 | — | — | 2026-02-20 |
| `situacao` | string | 456 | — | — | Ativo |
| `motivo_evasao` | str | 96 | — | — | Problemas de transporte |


# Diagnóstico da Qualidade dos Dados

#### a) Verifique a existência de valores ausentes, apresentando a quantidade e o percentual de valores ausentes em cada atributo;

In [5]:
null_summary = pd.DataFrame({
    "Qtd. Ausentes": df.isna().sum(),
    "% Ausentes": df.isna().mean() * 100
})

print(null_summary)

                       Qtd. Ausentes  % Ausentes
id_aluno                           0         0.0
idade                             12         2.4
escola_origem                      0         0.0
municipio_residencia              12         2.4
estado_residencia                  0         0.0
renda_familiar_mensal             22         4.4
pessoas_domicilio                  0         0.0
acesso_internet                   14         2.8
trabalha                           0         0.0
turno                              0         0.0
horas_estudo_semana               17         3.4
frequencia_percentual             18         3.6
nota_media                        16         3.2
reprovacoes                        0         0.0
data_matricula                    12         2.4
situacao                           0         0.0
motivo_evasao                    398        79.6


#### b) Verifique a existência de duplicidades;

In [6]:
print(df.describe()["id_aluno"])

count       500
unique      470
top       A0322
freq          2
Name: id_aluno, dtype: object


Há 30 ids repetidos, sendo eles: 

In [7]:
ids_repetidos = df.loc[
    df["id_aluno"].duplicated(), 'id_aluno'
].unique().tolist()

print(ids_repetidos)

['A0326', 'A0311', 'A0120', 'A0093', 'A0127', 'A0441', 'A0087', 'A0256', 'A0013', 'A0100', 'A0429', 'A0151', 'A0353', 'A0126', 'A0021', 'A0230', 'A0440', 'A0272', 'A0342', 'A0397', 'A0169', 'A0052', 'A0436', 'A0136', 'A0199', 'A0322', 'A0233', 'A0467', 'A0082', 'A0376']


#### c) Verifique a existência de Inconsistências, identificando em seu relatório, pelo menos, três atributos que apresentem diferentes formas de representar a mesma informação. Exemplo: Maceio, maceio, MACEIÓ, Maceió.

| Problema | Atributo | Explicação |
|---|---|---|
| **Consistência** | **Idade** | Duas linhas fogem do padrão: `19 anos` e `vinte`. |
| **Consistência** | **Escola Origem** | 7 valores únicos, embora existam apenas 2 classificações. |
| **Consistência** | **Município Residência** | 9 valores únicos, mas apenas 6 municípios. |
| **Consistência** | **Pessoas Domicílio** | Um valor está escrito por extenso. |
| **Consistência** | **Acesso Internet** | 6 valores únicos para apenas `Sim` ou `Não`. |
| **Consistência** | **Trabalha** | Existem diferentes representações da mesma classificação. |
| **Consistência** | **Horas Estudo Semana** | Alguns valores incluem a palavra `horas`. |
| **Consistência** | **Frequência Percentual** | Diferentes formatos de representação da porcentagem. |
| **Consistência** | **Nota Média** | Um valor está escrito por extenso. |
| **Consistência** | **Reprovações** | Um valor está escrito por extenso. |
| **Consistência** | **Data Matrícula** | Mistura `2026-02-17` e `02/17/2026`. |
| **Consistência** | **Situação** | `Evadido` e `Evadiu` representam a mesma situação. |


#### d) Identifique valores inválidos ou suspeitos, que não estejam de acordo com o domínio esperado. Exemplos: idade = -3, frequência = 120, nota = 15.

| Problema | Atributo | Explicação |
|---|---|---|
| **Validade** | **Idade** | Uma linha apresenta o valor `145`. |
| **Validade** | **Renda Familiar Mensal** | Uma linha apresenta um valor negativo. |
| **Validade** | **Frequência Percentual** | Há linhas com valores acima de `100` e uma linha com valor negativo. |
| **Validade** | **Nota Média** | Há valores acima de `10` e um valor negativo. |

# Limpeza e Transformação

#### a) Correção/remoção de duplicidades – apresente um relatório informando a quantidade que havia antes, a quantidade removida e o total de registros após a operação de limpeza.

In [8]:
df = df.drop_duplicates()

print(df.describe()["id_aluno"])

count       470
unique      470
top       A0275
freq          1
Name: id_aluno, dtype: object


Com o código acima, removemos as 30 duplicatas identificadas anteriormente.

#### b) Realize a padronização de atributos categóricos que possuam diferenças de letras maiúsculas/minúsculas, espaços, acentuação, abreviações ou representações diferentes.

In [9]:
col = df["escola_origem"].astype("string").str.strip().str.upper()

df["escola_origem"] = col.replace({
    "PUB.": "Pública",
    "PUBLICA": "Pública",
    "ESCOLA PÚBLICA": "Pública",
    "ESCOLA PRIVADA": "Privada",
    "PARTICULAR": "Privada",
    "PÚBLICA": "Pública",
    "PRIVADA": "Privada"
})

In [ ]:
df["escola_origem"].unique()

In [ ]:
col = df["municipio_residencia"].astype("string").str.strip().str.upper()

df["municipio_residencia"] = col.replace({
    "MACEIO": "Maceió",
    "MACEIÓ": "Maceió",
    "MARECHAL DEODORO": "Marechal Deodoro",
    "ARAPIRACA": "Arapiraca",
    "PALMEIRA DOS INDIOS": "Palmeira dos Índios",
    "PALMEIRA DOS ÍNDIOS": "Palmeira dos Índios",
    "RIO LARGO": "Rio Largo",
    "UNIAO DOS PALMARES": "União dos Palmares",
    "UNIÃO DOS PALMARES": "União dos Palmares"
})

In [ ]:
df["municipio_residencia"].unique()

In [ ]:
col = df["acesso_internet"].astype("string").str.strip().str.upper()

df["acesso_internet"] = col.replace({
    "1": "Sim",
    "S": "Sim",
    "SIM": "Sim",
    "YES": "Sim",
    "NÃO": "Não",
    "N": "Não",
    "NO": "Não"
})

In [ ]:
df["acesso_internet"].unique()

In [ ]:
col = df["turno"].astype("string").str.strip().str.upper()

df["turno"] = col.replace({
    "MANHÃ": "Manhã",
    "M": "Manhã",
    "MANHA": "Manhã",
    "TARDE": "Tarde",
    "T": "Tarde",
    "NOITE": "Noite",
    "N": "Noite"
})

In [ ]:
df["turno"].unique()

In [ ]:
col = df["situacao"].astype("string").str.strip().str.upper()

df["situacao"] = col.replace({
    "ATIVO": "Ativo",
    "EVADIDO": "Evadido",
    "EVADIU": "Evadido",
    "DESISTENTE": "Evadido",
    "MATRICULADO": "Matriculado",
})

In [ ]:
df["situacao"].unique()

#### c) Verifique e corrija atributos numéricos armazenados como texto.

In [ ]:
df["idade"] = df["idade"].replace({
    "vinte": "20",
    "20 anos": "20",
    "19 anos": "19"
})
df["idade"] = pd.to_numeric(df["idade"], errors="coerce")
df = df.dropna(subset=["idade"])
df = df[(df["idade"] > 0) & (df["idade"] < 100)]
df["idade"] = df["idade"].astype(int)

In [ ]:
generate_gaussian_distribution(df["idade"], "Idade", "Frequência", "Distribuição da idade")

Como não são valores bem distribuídos, e nem a presença de outliers, optei por remover a os valores nulos.

In [ ]:
df["renda_familiar_mensal"] = (
    df["renda_familiar_mensal"]
        .str.replace("R$", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
)

df["renda_familiar_mensal"] = pd.to_numeric(df["renda_familiar_mensal"], errors="coerce")

In [ ]:
df["pessoas_domicilio"] = df["pessoas_domicilio"].replace({
    "quatro": "4",
})

df["pessoas_domicilio"] = pd.to_numeric(df["pessoas_domicilio"], errors="coerce")

In [ ]:
horas_estudo_semana = (
    df["horas_estudo_semana"]
        .str.replace("horas", "", regex=False)
        .str.replace("h", "", regex=False)
        .str.strip()
)

horas_estudo_semana = pd.to_numeric(df["horas_estudo_semana"], errors="coerce")

In [ ]:
generate_gaussian_distribution(horas_estudo_semana, "Horas de estudo", "Frequência", "Distribuição das horas de estudo")

Como os dados são numéricos e bem distribuídos, optarei por tratar os valores nulos de horas_estudo com uma inputação estatística, aplicando a média.

In [ ]:
round(horas_estudo_semana.mean(), 1)

In [ ]:
df["horas_estudo_semana"] = (
    df["horas_estudo_semana"]
        .str.replace("horas", "", regex=False)
        .str.replace("h", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
)

df["horas_estudo_semana"] = df["horas_estudo_semana"].fillna(8.4)

df["horas_estudo_semana"] = pd.to_numeric(df["horas_estudo_semana"], errors="coerce")

In [ ]:
frequencia_percentual = df["frequencia_percentual"].replace({
    "oitenta": "80",
    "85%": "85",
})

frequencia_percentual = pd.to_numeric(df["idade"], errors="coerce")

In [ ]:
generate_gaussian_distribution(frequencia_percentual, "Frequência percentual", "Frequência", "Distribuição da frequência percentual")

In [ ]:
frequencia_percentual.median()

Apesar do percentual de nulos para essa coluna ser de 3.6%, eu não gostaria de descartar os valores nulos, prefiro optar por uma imputação estatística, pois das 18 linhas com valores nulos para frequência percentual, 4 delas possuem evasão justificada, o que eu acho interessante para a análise. A frequência percentual não é bem distribuída, portanto, estarei aplicando a mediana nos valores nulos.

In [ ]:
df["frequencia_percentual"] = df["frequencia_percentual"].replace({
    "oitenta": "80",
    "85%": "85",
})

df["frequencia_percentual"] = df["frequencia_percentual"].fillna(20)

df["frequencia_percentual"] = pd.to_numeric(df["frequencia_percentual"], errors="coerce")

In [ ]:
get_invalid_numbers(df["nota_media"])

In [ ]:
nota_media = df["nota_media"].replace({
    "dez": "10",
    "8,5": "8.5"
})

nota_media = pd.to_numeric(nota_media, errors="coerce")

generate_gaussian_distribution(nota_media, "Nota média", "Frequência", "Distribuição da nota média")

In [ ]:
round(nota_media.median(), 1)

A nota média é bem distribuída, porém tem alguns outliers acima de 10, então vamos aplicar a mediana para os valores nulos novamente.

In [ ]:
df["nota_media"] = df["nota_media"].replace({
    "dez": "10",
    "8,5": "8.5"
})

df["nota_media"] = df["nota_media"].fillna(7.2)

df["nota_media"] = pd.to_numeric(df["nota_media"], errors="coerce")

In [ ]:
df["nota_media"].isna().sum()

In [ ]:
get_invalid_numbers(df["reprovacoes"])

In [ ]:
df["reprovacoes"] = df["reprovacoes"].replace({
    "duas": "2",
})

df["reprovacoes"] = pd.to_numeric(df["reprovacoes"], errors="coerce")

In [ ]:
df["reprovacoes"].isna().sum()

#### d) Padronize data_matricula para um único formato de data. Além disso, valores que não representam datas válidas devem ser identificados.



Formatos possíveis

Ano - Mês - Dia
Dia - Mês - Ano

Dia / Mês / Ano
Ano / Mês / Dia

Dia / Mês / Ano (Formato Reduzido, 26 ao invés de 2026)

In [ ]:
def format_date(date):
    if pd.isna(date):
        return None

    if '-' in date:
        parts = date.split('-')
    elif '/' in date:
        parts = date.split('/')
    else:
        return None

    if len(parts) != 3:
        return None

    # YYYY-MM-DD
    if len(parts[0]) == 4:
        year, month, day = parts

    # DD-MM-YYYY
    elif len(parts[2]) == 4:
        day, month, year = parts

    # DD-MM-YY
    elif len(parts[2]) == 2:
        day, month, year = parts
        year = "20" + year

    else:
        return None

    # If month > 12, day and month were probably swapped
    if int(month) > 12:
        month, day = day, month

    return f"{year}-{month}-{day}"

In [ ]:
df["data_matricula"] = df["data_matricula"].apply(format_date)

In [ ]:
converted = pd.to_datetime(
    df["data_matricula"],
    format="%Y-%m-%d",
    errors="coerce"
)

failed = df.loc[
    converted.isna() & df["data_matricula"].notna(),
    "data_matricula"
]

print(failed)

Datas não válidas no dataset.

In [ ]:
df["data_matricula"] = pd.to_datetime(
    df["data_matricula"],
    format="%Y-%m-%d",
    errors="coerce"
)

In [ ]:
df["data_matricula"].isna().sum()

#### e) Identifique atributos que utilizam escalas diferentes para representar a mesma informação. Padronize todos os registros para uma única escala.

In [ ]:
df["nota_media"][df["nota_media"] > 10]

In [ ]:
df["nota_media"] = df["nota_media"].where(
    df["nota_media"] <= 10,
    df["nota_media"] / 10
)

Dividi por 10 as notas acima de 10, pois elas estão claramente em uma escala diferente das outras, algo que eu identifico com 0-100, ao invés de 0-10.

In [ ]:
df["nota_media"][df["nota_media"] > 10]

In [ ]:
df["frequencia_percentual"][df["frequencia_percentual"] <= 1]

In [ ]:
df["frequencia_percentual"] = df["frequencia_percentual"].abs()

In [ ]:
df["frequencia_percentual"][df["frequencia_percentual"] <= 1]

É um pouco estranho ter frequências igual ou menos que 1%, porque é literalmente difícil faltar tanto, uma coisa é ter 0% de frequência, mas 0.901 é um número estranho. Por exemplo, no curso de IA, por matéria, temos 96 aulas (4 aulas por dia, 4 dias por mês, 6 semestres), para faltar 1% das aulas, era preciso ir a 0.96 delas, o que é impossível. Se nós considerarmos todas as matérias do semestre, teríamos 384 aulas, faltar 1% seria faltar 3,84 aulas, o que ainda seria impossível no nosso curso, pois como as aulas são no mesmo dia, você recebe 4 faltas ou 4 presenças. O motivo pelo qual eu decidi não tratar esses valores como escritos em uma escala diferente é porque, querendo ou não, eles ainda são plausíveis em vários contextos. Por exemplo, pela idade dos estudantes, eu posso assumir que é um contexto de faculdade, vamos supor que eles são estudantes de medicina e possuem 5 aulas por dia, isso daria 600 aulas por semestre, faltar 1% seria ter presença em apenas 6 aulas. Em um aluno desistente, isso seria possível. Tem vários cursos integrais com muitas aulas em que seria possível ter uma frequência tão baixa, então prefiro não ir no achismo e mudar a escala, no cenário das notas, era bem evidente pela própria relação entre as notas, mas aqui não. 

In [ ]:
df["frequencia_percentual"][df["frequencia_percentual"] > 100]

In [ ]:
df["frequencia_percentual"] = df["frequencia_percentual"].mask(
    df["frequencia_percentual"] > 100
)

In [ ]:
df["frequencia_percentual"][df["frequencia_percentual"] > 100]

In [ ]:
df["frequencia_percentual"] = df["frequencia_percentual"].fillna(20)

### 4. Tratamento de Valores Ausentes

a) Para cada atributo que apresentar valores ausentes, escolha uma estratégia de tratamento:
- manter o valor ausente;
- remover o registro;
- preencher com mediana;
- preencher com moda;
- criar categoria específica;
- outra estratégia justificada.

In [ ]:
null_summary = pd.DataFrame({
    "Qtd. Ausentes": df.isna().sum(),
    "% Ausentes": df.isna().mean() * 100
})

print(null_summary)

Como municipio_residencia é um dado categórico, vamos substituir seus valores nulos pela moda:

In [ ]:
df["municipio_residencia"].mode()

In [ ]:
df["municipio_residencia"] = df["municipio_residencia"].fillna("Maceió")

Para a renda_familiar_mensal, vamos analisar a distribuição gaussiana de seus valores.

In [ ]:
generate_gaussian_distribution(df["renda_familiar_mensal"], "Renda familiar mensal", "Frequência", "Distribuição da renda familiar mensal")

A renda familiar mensal é bem distribuída, portanto, vamos substituir os valores nulos pela média:

In [ ]:
round(df["renda_familiar_mensal"].mean(), 2)

In [ ]:
df["renda_familiar_mensal"] = df["renda_familiar_mensal"].fillna(3162.59)

acesso_internet é outro dado categórico, vamos ver se ele possui uma moda:

In [ ]:
df["acesso_internet"].mode()

Como ele possui uma moda, vamos aplicar o valor Sim nos valores nulos:

In [ ]:
df["acesso_internet"] = df["acesso_internet"].fillna("Sim")

Data matrícula é um dado mais categórico do que quantitativo, mas não é nenhum dos dois. Eu entendo que quando aplicamos esses métodos de substituir valores nulos, nós estamos adicionando valores que são próximos dos esperados para aquele conjunto de dados e que não vão influenciar nossa análise. Por exemplo, quando aplicamos a média da renda familiar nas famílias com valores nulos, é de se esperar que pessoas que frequentem a mesma escola/universidade vão ter, em sua maioria, rendas parecidas. A mesma coisa para o acesso à internet, há apenas 2 alunos com acesso marcado como não nesse dataset, logo é condizente com a realidade desses dados manter a moda nos valores nulos. Em data matrícula, por outro lado, é algo mais randômico, e não tão relevante para a análise de evasão escolar, por tanto, prefiro deixar esse campo em nulo.

Sobre o motivo_evasão, faz sentido ele ter tantos nulos, afinal ele é um campo em que só está presente quando o aluno evadiu. Nos alunos que evadiram, há 9 registros de motivo_evasão como nulo, o que eu acho que devem permanecer nulos, pois eu considero a falta de justificativa um dado importante para a análise final. 

In [ ]:
df.to_csv("dataset-final.csv", index=False)